# Sine–Gordon: kink–antikink collision

We solve $u_{tt}=u_{xx}-\sin u$ on $[-4,4]$, for $0\le t\le10$.
The exact two-soliton scattering solution is
$$u(x,t)=4\arctan\left[\frac{\sinh(\gamma c(t-t_c))}
{c\cosh(\gamma(x-x_c))}\right],\qquad \gamma=(1-c^2)^{-1/2},$$
with $c=0.6$, $t_c=5$, and $x_c=0.4$. The incoming kink and antikink
collide at $(x_c,t_c)$ and separate again. This is a scattering solution,
not a breather or a linear superposition of two single-kink profiles.
At collision $u=0$ everywhere, but $u_t\ne0$: the wave has not disappeared.
The continuous arctangent expression avoids a branch jump at collision.

Both endpoint values and initial displacement/velocity come from this exact
solution. The shifted collision center gives unequal endpoint values; the short
finite interval has substantial boundary activity. There is no periodic wrap.
See [the Durham two-soliton illustrations](https://maths.dur.ac.uk/users/P.E.Dorey/SOLITONS_2025_26/SGpictures/SG_Kinkantikink.html).

Install `python -m pip install -e '.[host,notebook,test]' -e './packages/models[precision,test]'` from the repository
root and select that kernel. Assembly and evolution use `pybspf` directly.


In [ ]:
import bspf_models.waves.sine_gordon as bspf_sine_gordon
import pybspf.calculus as bspf_calculus
import pybspf.galerkin as bspf_galerkin
import pybspf.operators as bspf_operators
import pybspf.plans as bspf_plans

import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
import pybspf as b

speed = 0.6
collision_time, collision_center = 5., 0.4
gamma = 1 / jnp.sqrt(1 - speed**2)
times = jnp.linspace(0., 10., 201)

def exact(x, t):
    ratio = (jnp.sinh(gamma*speed*(t-collision_time))
             / (speed*jnp.cosh(gamma*(x-collision_center))))
    return 4*jnp.arctan(ratio)

def exact_velocity(x, t):
    return jax.jvp(lambda tau: exact(x, tau), (t,), (jnp.ones_like(t),))[1]

def exact_gradient(x, t):
    return jax.jvp(lambda points: exact(points, t), (x,), (jnp.ones_like(x),))[1]

boundary = lambda t: exact(jnp.array([-4., 4.]), t)


## Resolved weak form and time-dependent lifting

Full BSPF trial functions are integrated with Gauss quadrature. Endpoint
samples are prescribed, while interior samples evolve by fourth-order RK4.
Writing $u=Q_iq+Q_bg(t)$ gives
$$M_{ii}\ddot q=-K_{ii}q-K_{ib}g
-Q_i^TW\sin(Q_iq+Q_bg)-M_{ib}\ddot g.$$
The solver obtains boundary velocity and acceleration by JAX differentiation.
It projects the sine term at quadrature points, rather than applying a nodal
nonlinearity to a dense mass matrix. Explicit RK4 requires a stable wave time
step; the solver does not choose one automatically.


In [ ]:
def solve(n=129, substeps=25, quadrature_order=8):
    x = jnp.linspace(-4., 4., n)
    plan = bspf_plans.plan_1d(x, degree=7, n_basis=24, boundary_points=9)
    weak = bspf_galerkin.galerkin_1d(plan, derivative_order=1,
                         quadrature_order=quadrature_order)
    u, v = bspf_sine_gordon.integrate_sine_gordon(
        weak, exact(x, times[0]), exact_velocity(x, times[0]), times,
        boundary=boundary, substeps=substeps)
    return x, plan, weak, u, v

x, plan, weak, u, v = solve()
reference = exact(x[None, :], times[:, None])
field_error = float(jnp.max(jnp.abs(u-reference)))
velocity_error = float(jnp.max(jnp.abs(v-exact_velocity(x[None, :], times[:, None]))))
boundary_error = float(jnp.max(jnp.abs(u[:, jnp.array([0, -1])]
                                        -jax.vmap(boundary)(times))))
print(f"Maximum displacement error: {field_error:.3e}")
print(f"Maximum velocity error:     {velocity_error:.3e}")
print(f"Dirichlet residual:         {boundary_error:.3e}")
print(f"Maximum endpoint mismatch: {float(jnp.max(jnp.abs(u[:, -1]-u[:, 0]))):.3f}")
assert field_error < 2e-8 and velocity_error < 2e-7
assert boundary_error < 1e-13


## Spatial, temporal and quadrature checks

Compare 65 and 129 samples with the same spline configuration. Halve the time
step from 0.002 to 0.001 and independently increase Gauss order from 8 to 10.
These checks separate spatial error from time stepping and under-integration;
they do not assert exponential convergence from two grids.


In [ ]:
xc, _, _, uc, _ = solve(n=65)
coarse_error = float(jnp.max(jnp.abs(uc-exact(xc[None, :], times[:, None]))))
_, _, _, ut, _ = solve(substeps=50)
_, _, _, uq, _ = solve(quadrature_order=10)
time_change = float(jnp.max(jnp.abs(ut-u)))
quadrature_change = float(jnp.max(jnp.abs(uq-u)))
print(f"65-point displacement error: {coarse_error:.3e}")
print(f"129-point error:             {field_error:.3e}")
print(f"Halved-step change:          {time_change:.3e}")
print(f"Gauss 8 -> 10 change:        {quadrature_change:.3e}")
assert coarse_error > 20*field_error
assert time_change < 1e-9 and quadrature_change < 1e-9


## Energy balance during collision

The finite-interval energy and boundary power satisfy
$$E=\int_{-4}^{4}\left[\tfrac12u_t^2+\tfrac12u_x^2+1-\cos u\right]dx,
\qquad E'=[u_tu_x]_{-4}^{4}.$$
Energy enters and leaves through the prescribed boundaries, so finite-interval
energy is not constant. The reference integrates the analytic two-soliton
energy density using independent 256-point Gauss–Legendre quadrature, checked
against 512 points. Numerical energy uses the BSPF weak-form quadrature.
The boundary-power integral uses BSPF on the output-time grid; its residual
also contains temporal-quadrature and endpoint-derivative errors.


In [ ]:
uq = u @ weak.values.T
vq = v @ weak.values.T
uxq = u @ weak.derivative_values.T
energy_density = 0.5*vq**2 + 0.5*uxq**2 + 1-jnp.cos(uq)
energy = energy_density @ weak.quadrature_weights

def reference_energy(order):
    nodes, weights = np.polynomial.legendre.leggauss(order)
    points = jnp.asarray(4*nodes)[None, :]
    t = times[:, None]
    density = (0.5*exact_velocity(points, t)**2
               +0.5*exact_gradient(points, t)**2+1-jnp.cos(exact(points, t)))
    return density @ jnp.asarray(4*weights)

exact_energy = reference_energy(256)
reference_change = float(jnp.max(jnp.abs(exact_energy-reference_energy(512))))
ux = bspf_operators.differentiate(plan, u.T).T
power = v[:, -1]*ux[:, -1] - v[:, 0]*ux[:, 0]
time_plan = bspf_plans.plan_1d(times, degree=7, n_basis=32, boundary_points=9)
transferred = bspf_calculus.antiderivative(time_plan, power)
energy_error = float(jnp.max(jnp.abs(energy-exact_energy)))
balance_error = float(jnp.max(jnp.abs(energy-energy[0]-transferred)))
print(f"Reference quadrature change: {reference_change:.3e}")
print(f"Energy reference error:      {energy_error:.3e}")
print(f"Integrated balance error:    {balance_error:.3e}")
print(f"Energy: {float(energy[0]):.6f} -> {float(jnp.max(energy)):.6f} -> {float(energy[-1]):.6f}")
assert reference_change < 1e-11
assert energy_error < 2e-7
assert balance_error < 2e-6


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8), constrained_layout=True)
for index in (0, 60, 100, 140, 200):
    line, = axes[0, 0].plot(x, u[index], label=f"t={float(times[index]):g}")
    axes[0, 0].plot(x, reference[index], "--", color=line.get_color(), alpha=.7)
axes[0, 0].set(xlabel="x", ylabel="u", title="Kink–antikink collision (dashed: exact)")
axes[0, 0].legend()
nodal_density = 0.5*v**2 + 0.5*ux**2 + 1-jnp.cos(u)
heat = axes[0, 1].pcolormesh(x, times, nodal_density, shading="auto", cmap="magma")
axes[0, 1].set(xlabel="x", ylabel="t", title="Energy density: approach, collision, separation")
fig.colorbar(heat, ax=axes[0, 1], label="Energy density")
axes[1, 0].semilogy(times[1:], np.max(np.abs(np.asarray(u-reference)), axis=1)[1:])
axes[1, 0].axvline(collision_time, color="gray", linestyle=":", label="Collision")
axes[1, 0].set(xlabel="t", ylabel="Maximum field error", title="Error over the full interval")
axes[1, 0].legend()
axes[1, 1].plot(times, energy, label="Numerical energy")
axes[1, 1].plot(times, exact_energy, "--", label="Exact reference")
axes[1, 1].plot(times, energy[0]+transferred, ":", label="Initial + boundary transfer")
axes[1, 1].set(xlabel="t", ylabel="Energy", title="Finite-interval energy balance")
axes[1, 1].legend()
plt.show()
